In [2]:
import requests
from zipfile import ZipFile
import io
import tempfile
from seviri_hdf5_retrieval import import_SEVIRI
from pathlib import Path
from seviri_hdf5_retrieval import generate_area_def
import xarray as xr
from requests import ConnectTimeout
from time import sleep


In [3]:
file_names = !eumdac search --collection EO:EUM:DAT:MSG:HRSEVIRI \
	--start	2024-03-01T00:00:00.000 \
	--end	2024-04-01T00:00:00.000

In [4]:
token_query = !eumdac token
print(token_query[0])
access_token = token_query[1]

The following token is valid until 2025-07-24 17:01:08.266771


In [5]:
from requests.exceptions import ChunkedEncodingError, ConnectionError

def request_data(file: str, access_token: str) -> requests.Response:
    url = "https://api.eumetsat.int/data/download/1.0.0/collections/EO%3AEUM%3ADAT%3AMSG%3AHRSEVIRI/products"
    request = f"{url}/{file}?access_token={access_token}"

    success = False
    while not success:
        try:
            print(f"Requesting '{file}'")
            response = requests.get(request, timeout=600)
            if response.status_code == 200:
                success = True
            else:
                print(f"Error code {response.status_code} encountered. Retrying...")
                sleep(10)
        except ConnectTimeout:
            print("Connection timed out. Retrying...")
        except ChunkedEncodingError:
            print("Connection broken: incomplete read. Retrying...")
        except ConnectionError:
            print("Couldn't connect, retrying...")
    return response

In [ ]:
from tqdm.notebook import tqdm

area_def = None
output_dir = "/data/WeatherGenerator/new_seviri/2024/03"

# use a temporary filesystem in RAM
#  this will reduce disk I/O and not wear out an SSD as much.
tmpdir = "/dev/shm"

min_lon = 3
max_lon = 8
min_lat = 50
max_lat = 54

for file in tqdm(file_names, smoothing=0.8, unit="imgs"):
    ncfile = f"{output_dir}/{file}.nc"
    if Path(ncfile).exists():
        print(f"File exists, skipping; {ncfile}")
        continue

    response = request_data(file, access_token)

    z = ZipFile(io.BytesIO(response.content))

    with tempfile.TemporaryDirectory(dir=tmpdir) as tmp:
        fname = Path(tmp) / f"{file}.nat"
        with fname.open(mode="w+b") as f:
            f.write(z.read(f"{file}.nat"))
        sc = import_SEVIRI(str(fname))
        if area_def is None:  # only generate area def once (slow!)
            area_def = generate_area_def(sc, min_lon, max_lon, min_lat, max_lat)
        rpj_scn = sc.resample(area_def, mode="nearest", retain_values=True)
        
        output_file = str(Path(tmp) / f"{file}.nc")
        rpj_scn.save_datasets(filename=output_file, engine="netcdf4")

        with xr.load_dataset(output_file, engine="netcdf4") as ds:
            ds = ds.drop_vars(["longitude", "latitude"])
            comp = dict(zlib=True, complevel=9) # compress data
            encoding = {var: comp for var in ds.data_vars}    
            ds.to_netcdf(ncfile, encoding=encoding)

  0%|          | 0/2976 [00:00<?, ?imgs/s]

File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331235741.966000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331234242.086000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331232742.179000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331231242.273000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331225742.366000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331224242.459000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331222742.553000000Z-NA.nc
File exists, skipping; /data/WeatherGenerator/new_seviri/2024/03/MSG3-SEVI-MSG15-0100-NA-20240331221242.646000000Z-NA.nc
File exists, skipping; /data/Wea

Rounding shape to (92, 115) and resolution from (0.04374789549931535, 0.043781785152837926) meters to (0.043478260869565216, 0.043478260869565216) meters


Requesting 'MSG3-SEVI-MSG15-0100-NA-20240305222742.751000000Z-NA'
Requesting 'MSG3-SEVI-MSG15-0100-NA-20240305221242.824000000Z-NA'
Requesting 'MSG3-SEVI-MSG15-0100-NA-20240305215741.691000000Z-NA'
Requesting 'MSG3-SEVI-MSG15-0100-NA-20240305214241.763000000Z-NA'
